# Deep Agents：原生 module 與 I/O

直接使用 `deepagents.create_deep_agent`。規劃工具由 `TodoListMiddleware` 明確加入。
檔案工具由 Deep Agents 提供。本教學不宣告額外的工具或呼叫封裝。

| 原生 API | 輸入 | 輸出 |
|---|---|---|
| `langchain_openai.ChatOpenAI` | 模型與 LiteLLM 設定 | 模型物件 |
| `deepagents.create_deep_agent` | 模型、middleware、系統提示詞 | `CompiledStateGraph` |
| `agent.ainvoke` | 含有 `messages` 的狀態字典 | 原生狀態字典 |
| `agent.astream` | 狀態與串流模式 | 原生狀態更新的非同步迭代器 |

輸出可包含 `messages`、`todos` 與 `files`。實際欄位取決於 middleware 與執行過程。
`files` 的值保留 SDK 的完整資料結構，不只取出文字。

本檔已保存一次真實執行的輸出。輸出中的識別碼、時間與用量只代表該次執行。


### 類別關係圖

`create_deep_agent` 是工廠函式。回傳值仍是 LangGraph 的可執行圖形。`TodoListMiddleware` 明確加入規劃工具。

執行下方區塊可顯示圖形。此區塊不會呼叫 LLM。


In [1]:
from IPython.display import HTML, display

display(HTML(r"""
<section style="background:#f8fafc;color:#172b4d;border:1px solid #dce4ef;border-radius:16px;padding:18px;margin:12px 0;font-family:system-ui,sans-serif;overflow-x:auto">
<h3 style="color:#172b4d;margin:0 0 12px;font-size:20px">類別關係圖</h3>
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1080 490" role="img" aria-label="Deep Agents 類別與工廠函式關係" style="display:block;width:100%;min-width:760px;height:auto;font-family:'Noto Sans CJK TC','Microsoft JhengHei',system-ui,sans-serif">
 <title>Deep Agents 類別與工廠函式關係</title><defs><marker id="langgraph-deep-agent-classes-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#64748b"/></marker></defs>
 <rect width="1080" height="490" fill="#f8fafc"/><rect x="25" y="25" width="300" height="158" rx="12" fill="#fff" stroke="#2563eb" stroke-width="1.6"/><path d="M26 73 H324" stroke="#dce4ef"/><text x="41" y="54" text-anchor="start" fill="#172b4d" font-size="18" font-weight="650">create_deep_agent()</text><text x="41" y="96" text-anchor="start" fill="#52647c" font-size="12" font-weight="400">deepagents · factory 函式</text><text x="41" y="125" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">model=ChatOpenAI(...)</text><text x="41" y="147" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">middleware=[TodoListMiddleware()]</text><rect x="405" y="25" width="280" height="158" rx="12" fill="#fff" stroke="#2563eb" stroke-width="1.6"/><path d="M406 73 H684" stroke="#dce4ef"/><text x="421" y="54" text-anchor="start" fill="#172b4d" font-size="18" font-weight="650">CompiledStateGraph</text><text x="421" y="96" text-anchor="start" fill="#52647c" font-size="12" font-weight="400">langgraph.graph.state</text><text x="421" y="125" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">ainvoke() → 狀態 dict</text><text x="421" y="147" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">astream() → 更新串流</text><rect x="765" y="25" width="290" height="158" rx="12" fill="#fff" stroke="#2563eb" stroke-width="1.6"/><path d="M766 73 H1054" stroke="#dce4ef"/><text x="781" y="54" text-anchor="start" fill="#172b4d" font-size="18" font-weight="650">ChatOpenAI</text><text x="781" y="96" text-anchor="start" fill="#52647c" font-size="12" font-weight="400">langchain_openai</text><text x="781" y="125" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">ainvoke(messages)</text><text x="781" y="147" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">回傳 AIMessage</text><rect x="25" y="300" width="300" height="158" rx="12" fill="#fff" stroke="#087e8b" stroke-width="1.6"/><path d="M26 348 H324" stroke="#dce4ef"/><text x="41" y="329" text-anchor="start" fill="#172b4d" font-size="18" font-weight="650">TodoListMiddleware</text><text x="41" y="371" text-anchor="start" fill="#52647c" font-size="12" font-weight="400">langchain.agents.middleware</text><text x="41" y="400" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">提供 write_todos 工具</text><text x="41" y="422" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">更新 todos</text><rect x="405" y="300" width="280" height="158" rx="12" fill="#fff" stroke="#087e8b" stroke-width="1.6"/><path d="M406 348 H684" stroke="#dce4ef"/><text x="421" y="329" text-anchor="start" fill="#172b4d" font-size="18" font-weight="650">StateBackend</text><text x="421" y="371" text-anchor="start" fill="#52647c" font-size="12" font-weight="400">deepagents.backends</text><text x="421" y="400" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">預設虛擬檔案 backend</text><text x="421" y="422" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">files[path] → 檔案資料</text><rect x="765" y="300" width="290" height="158" rx="12" fill="#fff" stroke="#087e8b" stroke-width="1.6"/><path d="M766 348 H1054" stroke="#dce4ef"/><text x="781" y="329" text-anchor="start" fill="#172b4d" font-size="18" font-weight="650">原生狀態 dict</text><text x="781" y="371" text-anchor="start" fill="#52647c" font-size="12" font-weight="400">欄位由圖形與 middleware 決定</text><text x="781" y="400" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">messages / todos / files</text><text x="781" y="422" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">files 保留內容與時間</text><polyline points="325,104 405,104" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-deep-agent-classes-arrow)"/><text x="365.0" y="90" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">建立</text><polyline points="685,104 765,104" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-deep-agent-classes-arrow)"/><text x="725.0" y="90" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">模型節點</text><polyline points="175,300 175,183" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-deep-agent-classes-arrow)"/><text x="235" y="249" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">加入建構</text><polyline points="545,183 545,300" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-deep-agent-classes-arrow)"/><text x="610" y="249" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">檔案工具使用</text><polyline points="685,380 765,380" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-deep-agent-classes-arrow)"/><text x="725.0" y="365" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">讀寫</text><polyline points="910,183 910,300" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-deep-agent-classes-arrow)"/><text x="980" y="249" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">訊息進入狀態</text></svg>
<p style="color:#52647c;line-height:1.7;margin:10px 0 0;font-size:14px">create_deep_agent 是工廠函式。回傳值仍是 LangGraph 的可執行圖形。TodoListMiddleware 明確加入規劃工具。</p>
</section>
"""))


## 1. 啟動與設定

在本資料夾執行：

```bash
uv sync --locked --group notebook
uv run --group notebook jupyter lab tutorial_native.ipynb
```

依序執行各區塊。推論與串流區塊會呼叫真實模型，並使用模型額度。
本教學直接使用頂層 `await`。每個串流範例會啟動另一個任務。
所有原生輸入與輸出都保留在 Notebook 變數中。


In [1]:
import os
import sys
from importlib.metadata import version
from pathlib import Path
from pprint import pprint

from dotenv import dotenv_values

assert sys.version_info[:2] == (3, 13), "請使用 Python 3.13 核心。"
project_dir = Path.cwd()
assert (project_dir / "pyproject.toml").is_file(), "請從本專案資料夾啟動 Notebook。"

# 直接載入設定。環境變數優先於專案 .env，再優先於根目錄 .env。
env = {
    **dotenv_values(project_dir.parent / ".env"),
    **dotenv_values(project_dir / ".env"),
    **os.environ,
}
endpoint = env["LITELLM_ENDPOINT"].strip().rstrip("/")
api_key = env["LITELLM_API_KEY"]
model_name = env["LITELLM_MODEL"]
assert endpoint.startswith(("http://", "https://")), "端點需要 HTTP 或 HTTPS 協定。"
assert api_key and model_name, "請設定金鑰與模型別名。"
print("Python：", sys.version.split()[0])
print("模型：", model_name)
# 請勿顯示 env、api_key 或含有金鑰的 client 設定。

Python： 3.13.15
模型： gpt-4.1-mini


## 2. 直接建立模型

設定工具呼叫所使用的模型。此步驟尚未推論。


In [2]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model=model_name,
    base_url=endpoint,
    api_key=api_key,
    use_responses_api=False,
    stream_usage=False,
    timeout=60,
    max_retries=0,
)
print(type(model))
print("langchain-openai：", version("langchain-openai"))

<class 'langchain_openai.chat_models.base.ChatOpenAI'>
langchain-openai： 1.7.3


## 3. 直接建立 Deep Agent

目前鎖定版本需要明確加入 `TodoListMiddleware`，才能確保 `write_todos` 可用。
預設檔案 backend 將虛擬檔案存放在執行狀態中。
這裡沒有 checkpointer，所以不同呼叫不會自動共用狀態。


In [3]:
from deepagents import create_deep_agent
from langchain.agents.middleware import TodoListMiddleware
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

agent = create_deep_agent(
    model=model,
    middleware=[TodoListMiddleware()],
    system_prompt=(
        "以繁體中文回答。使用短句。"
        "收到報告任務時，先用 write_todos 建立計畫。"
        "使用 write_file 寫入報告。完成後更新待辦事項。"
    ),
)
print("deepagents：", version("deepagents"))
print(type(agent))

deepagents： 0.7.32
<class 'langgraph.graph.state.CompiledStateGraph'>


### 控制流程圖

這是本例的概念流程。Deep Agents 會插入多個 middleware 節點。實際節點更新可在後面的 `astream` 輸出查看。

執行下方區塊可顯示圖形。此區塊不會呼叫 LLM。


In [2]:
from IPython.display import HTML, display

display(HTML(r"""
<section style="background:#f8fafc;color:#172b4d;border:1px solid #dce4ef;border-radius:16px;padding:18px;margin:12px 0;font-family:system-ui,sans-serif;overflow-x:auto">
<h3 style="color:#172b4d;margin:0 0 12px;font-size:20px">控制流程圖</h3>
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1080 600" role="img" aria-label="Deep Agents 工具與狀態控制流程" style="display:block;width:100%;min-width:760px;height:auto;font-family:'Noto Sans CJK TC','Microsoft JhengHei',system-ui,sans-serif">
 <title>Deep Agents 工具與狀態控制流程</title><defs><marker id="langgraph-deep-agent-flow-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#64748b"/></marker></defs>
 <rect width="1080" height="600" fill="#f8fafc"/><rect x="410" y="20" width="260" height="68" rx="12" fill="#f0f6ff" stroke="#2563eb" stroke-width="1.5"/><text x="540.0" y="48" text-anchor="middle" fill="#172b4d" font-size="16" font-weight="600">輸入狀態</text><text x="540.0" y="71" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">messages: [HumanMessage]</text><rect x="410" y="135" width="260" height="68" rx="12" fill="#f0f6ff" stroke="#2563eb" stroke-width="1.5"/><text x="540.0" y="163" text-anchor="middle" fill="#172b4d" font-size="16" font-weight="600">模型節點</text><text x="540.0" y="186" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">model.ainvoke(messages)</text><polygon points="540,245 690,310 540,375 390,310" fill="#fff7ed" stroke="#b45309" stroke-width="1.5"/><text x="540" y="308" text-anchor="middle" fill="#172b4d" font-size="16" font-weight="600">有 tool_calls？</text><rect x="765" y="277" width="280" height="68" rx="12" fill="#f0f6ff" stroke="#087e8b" stroke-width="1.5"/><text x="905.0" y="305" text-anchor="middle" fill="#172b4d" font-size="16" font-weight="600">工具與 middleware</text><text x="905.0" y="328" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">write_todos / write_file</text><rect x="765" y="400" width="280" height="68" rx="12" fill="#f0f6ff" stroke="#087e8b" stroke-width="1.5"/><text x="905.0" y="428" text-anchor="middle" fill="#172b4d" font-size="16" font-weight="600">更新原生狀態</text><text x="905.0" y="451" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">messages / todos / files</text><rect x="410" y="472" width="260" height="68" rx="12" fill="#f0f6ff" stroke="#2563eb" stroke-width="1.5"/><text x="540.0" y="500" text-anchor="middle" fill="#172b4d" font-size="16" font-weight="600">結束並回傳狀態</text><text x="540.0" y="523" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">messages / todos / files</text><polyline points="540,88 540,135" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-deep-agent-flow-arrow)"/><polyline points="540,203 540,245" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-deep-agent-flow-arrow)"/><polyline points="690,310 765,310" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-deep-agent-flow-arrow)"/><text x="727.5" y="295" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">是</text><polyline points="540,375 540,472" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-deep-agent-flow-arrow)"/><text x="565" y="425" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">否</text><polyline points="905,345 905,400" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-deep-agent-flow-arrow)"/><polyline points="1045,434 1062,434 1062,168 670,168" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-deep-agent-flow-arrow)"/><text x="870" y="154" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">回到模型</text><text x="28" y="570" text-anchor="start" fill="#52647c" font-size="14" font-weight="400">每個箭頭表示本例的一次狀態傳遞。模型可執行多輪工具呼叫。</text></svg>
<p style="color:#52647c;line-height:1.7;margin:10px 0 0;font-size:14px">這是本例的概念流程。Deep Agents 會插入多個 middleware 節點。實際節點更新可在後面的 astream 輸出查看。</p>
</section>
"""))


## 4. 顯示原生輸入，再直接執行

輸入是狀態字典。`messages` 裡面放置原生 `HumanMessage`。
模型負責提出工具呼叫。代理負責執行工具與繼續流程。
`/report.md` 是虛擬路徑，不是主機檔案。


In [4]:
agent_input = {
    "messages": [
        HumanMessage(
            content=(
                "先用 write_todos 建立一項待辦：寫入報告。"
                "用 write_file 將『LangGraph 使用節點與邊控制流程。』寫入 /report.md。"
                "將待辦狀態改為 completed，再簡短回答。"
            )
        )
    ]
}
config = {"recursion_limit": 40}
pprint(agent_input)
result = await agent.ainvoke(agent_input, config=config)
print(type(result))
print("原生狀態欄位：", list(result))
pprint(result)

{'messages': [HumanMessage(content='先用 write_todos 建立一項待辦：寫入報告。用 write_file 將『LangGraph 使用節點與邊控制流程。』寫入 /report.md。將待辦狀態改為 completed，再簡短回答。', additional_kwargs={}, response_metadata={})]}


<class 'dict'>
原生狀態欄位： ['messages', 'files', 'todos']
{'files': {'/report.md': {'content': 'LangGraph 使用節點與邊控制流程。',
                          'created_at': '2026-11-03T15:30:53.507337+00:00',
                          'encoding': 'utf-8',
                          'modified_at': '2026-11-03T15:30:53.507337+00:00'}},
 'messages': [HumanMessage(content='先用 write_todos 建立一項待辦：寫入報告。用 write_file 將『LangGraph 使用節點與邊控制流程。』寫入 /report.md。將待辦狀態改為 completed，再簡短回答。', additional_kwargs={}, response_metadata={}, id='b6ec1d7f-7b1c-4422-ac4d-58a08b012de1'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 3269, 'total_tokens': 3296, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 

## 5. 檢查原生訊息與工具請求

模型的工具請求位於 `AIMessage.tool_calls`。
工具結果是 `ToolMessage`。使用 `tool_call_id` 對應請求。
`usage_metadata` 是模型訊息的用量欄位；上游不一定回傳它。


In [5]:
for message in result["messages"]:
    print(type(message))
    pprint(message.model_dump())
    if isinstance(message, AIMessage):
        print("tool_calls：", message.tool_calls)
        print("usage_metadata：", message.usage_metadata)
    if isinstance(message, ToolMessage):
        print("tool_call_id：", message.tool_call_id)

print("最後訊息：", result["messages"][-1].content)

<class 'langchain_core.messages.human.HumanMessage'>
{'additional_kwargs': {},
 'content': '先用 write_todos 建立一項待辦：寫入報告。用 write_file 將『LangGraph '
            '使用節點與邊控制流程。』寫入 /report.md。將待辦狀態改為 completed，再簡短回答。',
 'id': 'b6ec1d7f-7b1c-4422-ac4d-58a08b012de1',
 'name': None,
 'response_metadata': {},
 'type': 'human'}
<class 'langchain_core.messages.ai.AIMessage'>
{'additional_kwargs': {'refusal': None},
 'content': '',
 'id': 'lc_run--01a1a208-d631-74b3-bcb0-afe870805c32-0',
 'invalid_tool_calls': [],
 'name': None,
 'response_metadata': {'finish_reason': 'tool_calls',
                       'id': 'chatcmpl-EgAlHAGQ7bJzshhmJgo21zn8L27yl',
                       'logprobs': None,
                       'model_name': 'gpt-4.1-mini',
                       'model_provider': 'openai',
                       'service_tier': 'default',
                       'system_fingerprint': 'fp_2a8f9b1a2d',
                       'token_usage': {'completion_tokens': 27,
                                 

## 6. 檢查原生 todos 與 files

`todos` 是待辦字典的清單。`content` 是項目內容，`status` 是狀態。
`files` 是路徑到檔案資料的映射。
此鎖定版本的檔案 `content` 是字串。檔案資料也可包含建立與修改時間。
直接顯示 SDK 回傳內容，可觀察版本差異。


In [6]:
print("todos 型別：", type(result.get("todos")))
pprint(result.get("todos"))
print("files 型別：", type(result.get("files")))
pprint(result.get("files"))

if "/report.md" in result.get("files", {}):
    file_data = result["files"]["/report.md"]
    print("原生檔案欄位：", list(file_data))
    print("content 型別：", type(file_data["content"]))
    pprint(file_data)

todos 型別： <class 'list'>
[{'content': '寫入報告', 'status': 'completed'}]
files 型別： <class 'dict'>
{'/report.md': {'content': 'LangGraph 使用節點與邊控制流程。',
                'created_at': '2026-11-03T15:30:53.507337+00:00',
                'encoding': 'utf-8',
                'modified_at': '2026-11-03T15:30:53.507337+00:00'}}
原生檔案欄位： ['content', 'encoding', 'created_at', 'modified_at']
content 型別： <class 'str'>
{'content': 'LangGraph 使用節點與邊控制流程。',
 'created_at': '2026-11-03T15:30:53.507337+00:00',
 'encoding': 'utf-8',
 'modified_at': '2026-11-03T15:30:53.507337+00:00'}


### 循序圖

寫入待辦與檔案可能分成多輪。`files` 保留檔案資料；`messages` 保留模型與工具訊息。

執行下方區塊可顯示圖形。此區塊不會呼叫 LLM。


In [3]:
from IPython.display import HTML, display

display(HTML(r"""
<section style="background:#f8fafc;color:#172b4d;border:1px solid #dce4ef;border-radius:16px;padding:18px;margin:12px 0;font-family:system-ui,sans-serif;overflow-x:auto">
<h3 style="color:#172b4d;margin:0 0 12px;font-size:20px">循序圖</h3>
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1080 552" role="img" aria-label="Deep Agents 工具與狀態更新的循序圖" style="display:block;width:100%;min-width:760px;height:auto;font-family:'Noto Sans CJK TC','Microsoft JhengHei',system-ui,sans-serif">
 <title>Deep Agents 工具與狀態更新的循序圖</title><defs><marker id="langgraph-deep-agent-sequence-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#64748b"/></marker></defs>
 <rect width="1080" height="552" fill="#f8fafc"/><rect x="16.0" y="15" width="188" height="62" rx="10" fill="#fff" stroke="#2563eb"/><text x="110.0" y="41" text-anchor="middle" fill="#172b4d" font-size="16" font-weight="600">Notebook</text><text x="110.0" y="62" text-anchor="middle" fill="#52647c" font-size="11" font-weight="400">使用者程式</text><line x1="110.0" y1="77" x2="110.0" y2="502" stroke="#b6c4d6" stroke-dasharray="5 5"/><rect x="302.6666666666667" y="15" width="188" height="62" rx="10" fill="#fff" stroke="#2563eb"/><text x="396.6666666666667" y="41" text-anchor="middle" fill="#172b4d" font-size="16" font-weight="600">Deep Agent</text><text x="396.6666666666667" y="62" text-anchor="middle" fill="#52647c" font-size="11" font-weight="400">CompiledStateGraph</text><line x1="396.6666666666667" y1="77" x2="396.6666666666667" y2="502" stroke="#b6c4d6" stroke-dasharray="5 5"/><rect x="589.3333333333334" y="15" width="188" height="62" rx="10" fill="#fff" stroke="#2563eb"/><text x="683.3333333333334" y="41" text-anchor="middle" fill="#172b4d" font-size="16" font-weight="600">ChatOpenAI</text><text x="683.3333333333334" y="62" text-anchor="middle" fill="#52647c" font-size="11" font-weight="400">模型介面</text><line x1="683.3333333333334" y1="77" x2="683.3333333333334" y2="502" stroke="#b6c4d6" stroke-dasharray="5 5"/><rect x="876.0" y="15" width="188" height="62" rx="10" fill="#fff" stroke="#2563eb"/><text x="970.0" y="41" text-anchor="middle" fill="#172b4d" font-size="16" font-weight="600">工具 / middleware</text><text x="970.0" y="62" text-anchor="middle" fill="#52647c" font-size="11" font-weight="400">todos 與虛擬檔案</text><line x1="970.0" y1="77" x2="970.0" y2="502" stroke="#b6c4d6" stroke-dasharray="5 5"/><polyline points="110.0,115 396.6666666666667,115" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-deep-agent-sequence-arrow)"/><text x="253.33333333333334" y="105" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">1. ainvoke({messages}, config)</text><polyline points="396.6666666666667,164 683.3333333333334,164" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-deep-agent-sequence-arrow)"/><text x="540.0" y="154" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">2. 傳入任務與工具描述</text><polyline points="683.3333333333334,213 396.6666666666667,213" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-deep-agent-sequence-arrow)" stroke-dasharray="6 4"/><text x="540.0" y="203" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">3. AIMessage：工具請求</text><polyline points="396.6666666666667,262 970.0,262" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-deep-agent-sequence-arrow)"/><text x="683.3333333333334" y="252" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">4. write_todos / write_file</text><polyline points="970.0,311 396.6666666666667,311" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-deep-agent-sequence-arrow)" stroke-dasharray="6 4"/><text x="683.3333333333334" y="301" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">5. ToolMessage ＋ 狀態更新</text><polyline points="396.6666666666667,360 683.3333333333334,360" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-deep-agent-sequence-arrow)"/><text x="540.0" y="350" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">6. 傳入工具執行結果</text><polyline points="683.3333333333334,409 396.6666666666667,409" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-deep-agent-sequence-arrow)" stroke-dasharray="6 4"/><text x="540.0" y="399" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">7. 最終 AIMessage</text><polyline points="396.6666666666667,458 110.0,458" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-deep-agent-sequence-arrow)" stroke-dasharray="6 4"/><text x="253.33333333333334" y="448" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">8. dict：messages / todos / files</text><text x="30" y="530" text-anchor="start" fill="#52647c" font-size="13" font-weight="400">實線：呼叫或請求　　虛線：回傳資料或事件</text></svg>
<p style="color:#52647c;line-height:1.7;margin:10px 0 0;font-size:14px">寫入待辦與檔案可能分成多輪。files 保留檔案資料；messages 保留模型與工具訊息。</p>
</section>
"""))


## 7. 直接接收狀態更新串流

`create_deep_agent` 回傳 LangGraph 圖形，因此可直接呼叫 `astream`。
此處使用 `updates` 模式與 `v1` 格式。每個 chunk 以節點名稱為鍵。
middleware 也可能產生更新。chunk 的欄位不一定只有 `messages`。
此呼叫是另一個獨立任務，不會讀取前一個 `result`。


In [7]:
updates = []
async for chunk in agent.astream(
    {"messages": [HumanMessage(content="直接以一句話說明代理的用途。本次不用規劃或寫檔。")]},
    config={"recursion_limit": 20},
    stream_mode="updates",
    version="v1",
):
    updates.append(chunk)
    print(type(chunk))
    pprint(chunk)

<class 'dict'>
{'PatchToolCallsMiddleware.before_agent': None}


<class 'dict'>
{'model': {'messages': [AIMessage(content='代理的用途是代表使用者在系統或網路中執行特定任務或操作。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 3234, 'total_tokens': 3259, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 2816, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini', 'system_fingerprint': 'fp_2a8f9b1a2d', 'id': 'chatcmpl-EgAlMTDtvFsbnW3engTR7cS98iO7a', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1a208-eaa8-7760-b565-e03bac6acd11-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 3234, 'output_tokens': 25, 'total_tokens': 3259, 'input_token_details': {'audio': 0, 'cache_read': 2816}, 'output_token_details':

## 官方來源與版本

- [Deep Agents Quickstart](https://docs.langchain.com/oss/python/deepagents/quickstart)：建立與呼叫代理。
- [Backend](https://docs.langchain.com/oss/python/deepagents/backends)：狀態中的虛擬檔案。
- [LangGraph 串流](https://docs.langchain.com/oss/python/langgraph/streaming)：原生串流模式。

以專案的 `uv.lock` 為準。模型產生的工具順序與文字可能不同。
